In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost requests gradio joblib

In [ ]:
import os
import time
import joblib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from getpass import getpass
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

print("All libraries imported successfully.")

In [ ]:
GROQ_API_KEY = getpass(
    "Enter your Groq API key: "
)

if GROQ_API_KEY.strip():
    print("Groq API key loaded.")
else:
    print("Groq API key not provided.")

In [ ]:
OPEN_METEO_URL = (
    "https://api.open-meteo.com/v1/forecast"
)

GROQ_URL = (
    "https://api.groq.com/openai/v1/chat/completions"
)

GROQ_MODEL = (
    "openai/gpt-oss-120b"
)

REQUEST_TIMEOUT = 20

print("Configuration ready.")

In [ ]:
KP_DISTRICTS = {

    "Peshawar": (34.0151, 71.5249),
    "Charsadda": (34.1454, 71.7409),
    "Nowshera": (34.0153, 71.9747),
    "Mardan": (34.1989, 72.0403),
    "Swabi": (34.1200, 72.4700),

    "Swat": (35.2227, 72.4258),
    "Buner": (34.4833, 72.5333),
    "Shangla": (34.8700, 72.6500),
    "Malakand": (34.5700, 71.9300),
    "Lower Dir": (34.7700, 71.8700),
    "Upper Dir": (35.2100, 71.8800),

    "Abbottabad": (34.1688, 73.2215),
    "Mansehra": (34.3300, 73.2000),
    "Haripur": (33.9964, 72.9347),

    "Battagram": (34.6800, 73.0200),

    "Kohat": (33.5900, 71.4400),
    "Hangu": (33.5300, 71.0600),
    "Karak": (33.1200, 71.0900),

    "Bannu": (32.9900, 70.6000),
    "Lakki Marwat": (32.6100, 70.9100),

    "Dera Ismail Khan": (31.8300, 70.9000),
    "Tank": (32.2200, 70.3800)
}

print(
    f"{len(KP_DISTRICTS)} districts loaded."
)

In [ ]:
def get_weather(lat, lon):

    params = {

        "latitude": lat,
        "longitude": lon,

        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "soil_moisture_0_to_1cm",
            "wind_speed_10m"
        ],

        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum"
        ],

        "past_days": 7,
        "forecast_days": 2,

        "timezone": "auto"
    }

    response = requests.get(
        OPEN_METEO_URL,
        params=params,
        timeout=REQUEST_TIMEOUT
    )

    response.raise_for_status()

    return response.json()


print("Weather function created.")

In [ ]:
lat, lon = KP_DISTRICTS["Peshawar"]

weather_data = get_weather(
    lat,
    lon
)

print("Weather API working.")
print(
    weather_data.keys()
)

In [ ]:
def create_weather_features(data):

    hourly = data["hourly"]
    daily = data["daily"]

    temperature = pd.Series(
        hourly["temperature_2m"]
    ).dropna()

    humidity = pd.Series(
        hourly["relative_humidity_2m"]
    ).dropna()

    precipitation = pd.Series(
        hourly["precipitation"]
    ).dropna()

    soil = pd.Series(
        hourly["soil_moisture_0_to_1cm"]
    ).dropna()

    wind = pd.Series(
        hourly["wind_speed_10m"]
    ).dropna()

    daily_rain = pd.Series(
        daily["precipitation_sum"]
    ).dropna()

    daily_max_temp = pd.Series(
        daily["temperature_2m_max"]
    ).dropna()

    features = {

        "rainfall_24h":
            precipitation.tail(24).sum(),

        "rainfall_3day":
            precipitation.tail(72).sum(),

        "rainfall_7day":
            precipitation.sum(),

        "max_daily_rain":
            daily_rain.max(),

        "forecast_rain":
            daily_rain.tail(2).sum(),

        "max_temperature":
            daily_max_temp.max(),

        "average_temperature":
            temperature.mean(),

        "average_humidity":
            humidity.mean(),

        "soil_moisture":
            soil.mean(),

        "max_wind_speed":
            wind.max()
    }

    return features

In [ ]:
features = create_weather_features(
    weather_data
)

pd.DataFrame(
    [features]
)

In [ ]:
from google.colab import files

uploaded = files.upload()

print(
    "Dataset uploaded."
)

In [ ]:
DATASET_PATH = (
    "multi_hazard_kp.csv"
)

df = pd.read_csv("/content/multi_hazard_kp.csv")

print(
    "Dataset shape:",
    df.shape
)

df.head()

In [ ]:
print("Columns:")
print(
    df.columns.tolist()
)

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(
    df.isnull().sum()
)

In [ ]:
targets = [
    "heavy_rain",
    "heatwave",
    "flood"
]

for target in targets:

    print(
        f"\n{target}:"
    )

    print(
        df[target].value_counts()
    )

In [ ]:
FEATURES = [

    "rainfall_24h",
    "rainfall_3day",
    "rainfall_7day",
    "max_daily_rain",
    "forecast_rain",
    "max_temperature",
    "average_temperature",
    "average_humidity",
    "soil_moisture",
    "max_wind_speed"
]

print(
    "Number of features:",
    len(FEATURES)
)

In [ ]:
required_columns = (
    FEATURES + targets
)

missing_columns = [

    column

    for column in required_columns

    if column not in df.columns
]

if missing_columns:

    print(
        "Missing columns:"
    )

    print(
        missing_columns
    )

else:

    print(
        "All required columns found."
    )

In [ ]:
model_df = df[
    FEATURES + targets
].copy()

for column in FEATURES:

    model_df[column] = pd.to_numeric(
        model_df[column],
        errors="coerce"
    )

for target in targets:

    model_df[target] = pd.to_numeric(
        model_df[target],
        errors="coerce"
    )

print(
    model_df.isnull().sum()
)

In [ ]:
for column in FEATURES:

    model_df[column] = (
        model_df[column]
        .fillna(
            model_df[column].median()
        )
    )

for target in targets:

    model_df = model_df.dropna(
        subset=[target]
    )

    model_df[target] = (
        model_df[target]
        .astype(int)
    )

print(
    "Missing values handled."
)

In [ ]:
def train_models(
    data,
    target
):

    X = data[
        FEATURES
    ]

    y = data[
        target
    ]


    X_train, X_test, y_train, y_test = (
        train_test_split(

            X,
            y,

            test_size=0.20,

            random_state=42,

            stratify=y
        )
    )


    logistic = Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(
                max_iter=2000
            )
        )
    ])


    random_forest = (
        RandomForestClassifier(

            n_estimators=300,

            class_weight="balanced",

            random_state=42,

            n_jobs=-1
        )
    )


    xgboost = XGBClassifier(

        n_estimators=300,

        learning_rate=0.05,

        max_depth=5,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="logloss",

        random_state=42
    )


    models = {

        "Logistic Regression":
            logistic,

        "Random Forest":
            random_forest,

        "XGBoost":
            xgboost
    }


    for name, model in models.items():

        model.fit(
            X_train,
            y_train
        )


    return (
        models,
        X_train,
        X_test,
        y_train,
        y_test
    )

In [ ]:
def evaluate_models(
    models,
    X_test,
    y_test
):

    results = []

    for name, model in models.items():

        prediction = model.predict(
            X_test
        )

        probability = (
            model.predict_proba(
                X_test
            )[:, 1]
        )


        results.append({

            "Model": name,

            "Accuracy":
                accuracy_score(
                    y_test,
                    prediction
                ),

            "Precision":
                precision_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "Recall":
                recall_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "F1":
                f1_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_test,
                    probability
                )
        })


    return (
        pd.DataFrame(results)
        .sort_values(
            "F1",
            ascending=False
        )
    )

In [ ]:
rain_models = train_models(
    model_df,
    "heavy_rain"
)

(
    rain_models_dict,
    rain_X_train,
    rain_X_test,
    rain_y_train,
    rain_y_test
) = rain_models


rain_results = evaluate_models(

    rain_models_dict,

    rain_X_test,

    rain_y_test
)

rain_results

In [ ]:
best_rain_name = (
    rain_results.iloc[0]["Model"]
)

best_rain_model = (
    rain_models_dict[
        best_rain_name
    ]
)

print(
    "Best Rain model:",
    best_rain_name
)

In [ ]:
heat_models = train_models(
    model_df,
    "heatwave"
)

(
    heat_models_dict,
    heat_X_train,
    heat_X_test,
    heat_y_train,
    heat_y_test
) = heat_models


heat_results = evaluate_models(

    heat_models_dict,

    heat_X_test,

    heat_y_test
)

heat_results

In [ ]:
best_heat_name = (
    heat_results.iloc[0]["Model"]
)

best_heat_model = (
    heat_models_dict[
        best_heat_name
    ]
)

print(
    "Best Heat model:",
    best_heat_name
)

In [ ]:
flood_models = train_models(
    model_df,
    "flood"
)

(
    flood_models_dict,
    flood_X_train,
    flood_X_test,
    flood_y_train,
    flood_y_test
) = flood_models


flood_results = evaluate_models(

    flood_models_dict,

    flood_X_test,

    flood_y_test
)

flood_results

In [ ]:
best_flood_name = (
    flood_results.iloc[0]["Model"]
)

best_flood_model = (
    flood_models_dict[
        best_flood_name
    ]
)

print(
    "Best Flood model:",
    best_flood_name
)

In [ ]:
print("RAIN MODELS")
display(rain_results)

print("\nHEAT MODELS")
display(heat_results)

print("\nFLOOD MODELS")
display(flood_results)

In [ ]:
def show_confusion_matrix(
    model,
    X_test,
    y_test,
    title
):

    predictions = model.predict(
        X_test
    )

    matrix = confusion_matrix(
        y_test,
        predictions
    )

    plt.figure(
        figsize=(5, 4)
    )

    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[
            "No",
            "Yes"
        ],
        yticklabels=[
            "No",
            "Yes"
        ]
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "Actual"
    )

    plt.title(
        title
    )

    plt.show()

In [ ]:
show_confusion_matrix(

    best_rain_model,

    rain_X_test,

    rain_y_test,

    "Heavy Rain Confusion Matrix"
)

In [ ]:
show_confusion_matrix(

    best_heat_model,

    heat_X_test,

    heat_y_test,

    "Heatwave Confusion Matrix"
)

In [ ]:
show_confusion_matrix(

    best_flood_model,

    flood_X_test,

    flood_y_test,

    "Flood Confusion Matrix"
)

In [ ]:
def show_feature_importance(
    model,
    title
):

    if hasattr(
        model,
        "feature_importances_"
    ):

        importance = (
            model.feature_importances_
        )

    elif hasattr(
        model,
        "named_steps"
    ):

        print(
            "This model does not provide "
            "tree feature importance."
        )

        return

    else:

        print(
            "Feature importance unavailable."
        )

        return


    importance_df = pd.DataFrame({

        "Feature":
            FEATURES,

        "Importance":
            importance

    })


    importance_df = (
        importance_df
        .sort_values(
            "Importance",
            ascending=True
        )
    )


    plt.figure(
        figsize=(8, 5)
    )


    plt.barh(

        importance_df[
            "Feature"
        ],

        importance_df[
            "Importance"
        ]
    )


    plt.title(
        title
    )

    plt.xlabel(
        "Importance"
    )

    plt.tight_layout()

    plt.show()

    return importance_df

In [ ]:
flood_importance = (
    show_feature_importance(

        best_flood_model,

        "Flood Feature Importance"
    )
)

flood_importance

In [ ]:
joblib.dump(
    best_rain_model,
    "rain_model.pkl"
)

joblib.dump(
    best_heat_model,
    "heat_model.pkl"
)

joblib.dump(
    best_flood_model,
    "flood_model.pkl"
)

joblib.dump(
    FEATURES,
    "features.pkl"
)

print(
    "All models saved."
)

In [ ]:
def prepare_live_input(
    features
):

    row = pd.DataFrame(
        [features]
    )

    row = row[
        FEATURES
    ]

    return row

In [ ]:
def risk_level(
    probability
):

    if probability < 0.25:

        return "Low"

    elif probability < 0.50:

        return "Moderate"

    elif probability < 0.75:

        return "High"

    else:

        return "Severe"

In [ ]:
def predict_all_hazards(
    features
):

    input_data = (
        prepare_live_input(
            features
        )
    )


    rain_probability = float(

        best_rain_model
        .predict_proba(
            input_data
        )[0][1]
    )


    heat_probability = float(

        best_heat_model
        .predict_proba(
            input_data
        )[0][1]
    )


    flood_probability = float(

        best_flood_model
        .predict_proba(
            input_data
        )[0][1]
    )


    return {

        "heavy_rain": {

            "probability":
                rain_probability,

            "risk":
                risk_level(
                    rain_probability
                )
        },

        "heatwave": {

            "probability":
                heat_probability,

            "risk":
                risk_level(
                    heat_probability
                )
        },

        "flood": {

            "probability":
                flood_probability,

            "risk":
                risk_level(
                    flood_probability
                )
        }
    }

In [ ]:
lat, lon = KP_DISTRICTS[
    "Nowshera"
]

live_weather = get_weather(
    lat,
    lon
)

live_features = (
    create_weather_features(
        live_weather
    )
)

predictions = (
    predict_all_hazards(
        live_features
    )
)

predictions

In [ ]:
def explain_with_groq(
    district,
    features,
    predictions
):

    if not GROQ_API_KEY:

        return (
            "Groq explanation unavailable."
        )


    prompt = f"""
You are an AI assistant for an educational
multi-hazard environmental risk system.

District:
{district}

Machine-learning predictions:

Heavy Rain:
{predictions['heavy_rain']['probability']:.2%}
Risk:
{predictions['heavy_rain']['risk']}

Heatwave:
{predictions['heatwave']['probability']:.2%}
Risk:
{predictions['heatwave']['risk']}

Flood:
{predictions['flood']['probability']:.2%}
Risk:
{predictions['flood']['risk']}

Weather data:

7-day rainfall:
{features['rainfall_7day']:.2f} mm

24-hour rainfall:
{features['rainfall_24h']:.2f} mm

Forecast rainfall:
{features['forecast_rain']:.2f} mm

Maximum temperature:
{features['max_temperature']:.2f} C

Average humidity:
{features['average_humidity']:.2f} %

Soil moisture:
{features['soil_moisture']:.3f}

Wind:
{features['max_wind_speed']:.2f} km/h

Explain the results in simple language.

Mention which hazards have the highest
risk and which weather factors contributed.

Do not claim that a disaster will definitely
happen.

Clearly state that this is an ML-based
preliminary assessment and not an official
emergency warning.
"""


    headers = {

        "Authorization":
            f"Bearer {GROQ_API_KEY}",

        "Content-Type":
            "application/json"
    }


    payload = {

        "model":
            GROQ_MODEL,

        "messages": [

            {
                "role":
                    "system",

                "content":
                    "You explain ML environmental "
                    "risk results clearly."
            },

            {
                "role":
                    "user",

                "content":
                    prompt
            }
        ],

        "temperature":
            0.2,

        "max_tokens":
            250
    }


    response = requests.post(

        GROQ_URL,

        headers=headers,

        json=payload,

        timeout=30
    )


    response.raise_for_status()


    result = (
        response.json()
    )


    return (
        result[
            "choices"
        ][0][
            "message"
        ][
            "content"
        ].strip()
    )

In [ ]:
explanation = explain_with_groq(

    "Nowshera",

    live_features,

    predictions
)

print(
    explanation
)

In [ ]:
def run_hazard_assessment(
    district
):

    lat, lon = (
        KP_DISTRICTS[
            district
        ]
    )


    weather = get_weather(
        lat,
        lon
    )


    features = (
        create_weather_features(
            weather
        )
    )


    predictions = (
        predict_all_hazards(
            features
        )
    )


    explanation = (
        explain_with_groq(

            district,

            features,

            predictions
        )
    )


    return {

        "district":
            district,

        "features":
            features,

        "predictions":
            predictions,

        "explanation":
            explanation
    }

In [ ]:
result = (
    run_hazard_assessment(
        "Peshawar"
    )
)

print(
    result["predictions"]
)

print(
    "\nAI Explanation:\n"
)

print(
    result["explanation"]
)

In [ ]:
import gradio as gr

In [ ]:
def dashboard(district):
    try:
        result = run_hazard_assessment(district)
        predictions = result["predictions"]

        table = pd.DataFrame({
            "Hazard": ["🌧️ Heavy Rain", "🔥 Heatwave", "🌊 Flood"],
            "Probability": [
                predictions["heavy_rain"]["probability"],
                predictions["heatwave"]["probability"],
                predictions["flood"]["probability"]
            ],
            "Risk": [
                predictions["heavy_rain"]["risk"],
                predictions["heatwave"]["risk"],
                predictions["flood"]["risk"]
            ]
        })

        colors = {"Low": "#2ecc71", "Moderate": "#f1c40f", "High": "#e67e22", "Severe": "#e74c3c"}
        bar_colors = [colors[r] for r in table["Risk"]]

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar(table["Hazard"], table["Probability"], color=bar_colors)
        ax.set_ylim(0, 1)
        ax.set_ylabel("ML Probability")
        ax.set_title(f"Multi-Hazard Risk — {district}")
        fig.tight_layout()

        summary = (
            f"### 📍 {district} — Multi-Hazard Assessment\n\n"
            f"🌧️ Heavy Rain: **{predictions['heavy_rain']['risk']}** "
            f"({predictions['heavy_rain']['probability']:.1%})\n\n"
            f"🔥 Heatwave: **{predictions['heatwave']['risk']}** "
            f"({predictions['heatwave']['probability']:.1%})\n\n"
            f"🌊 Flood: **{predictions['flood']['risk']}** "
            f"({predictions['flood']['probability']:.1%})"
        )

        return summary, table, fig, result["explanation"]

    except Exception as e:
        error_msg = f"⚠️ Error while processing **{district}**:\n\n`{str(e)}`"
        empty_table = pd.DataFrame({"Hazard": [], "Probability": [], "Risk": []})
        empty_fig, ax = plt.subplots(figsize=(8, 5))
        ax.text(0.5, 0.5, "No data (error occurred)", ha="center", va="center")
        return error_msg, empty_table, empty_fig, "N/A"

In [ ]:
def compare_districts(selected_districts):
    if not selected_districts or len(selected_districts) < 2:
        empty_fig, ax = plt.subplots(figsize=(8, 5))
        ax.text(0.5, 0.5, "Select at least 2 districts to compare", ha="center", va="center")
        return "Select at least 2 districts.", pd.DataFrame(), empty_fig

    rows = []
    for d in selected_districts:
        try:
            lat, lon = KP_DISTRICTS[d]
            weather = get_weather(lat, lon)
            features = create_weather_features(weather)
            predictions = predict_all_hazards(features)

            rows.append({
                "District": d,
                "Heavy Rain": predictions["heavy_rain"]["probability"],
                "Heatwave": predictions["heatwave"]["probability"],
                "Flood": predictions["flood"]["probability"],
                "Highest Risk Hazard": max(
                    predictions,
                    key=lambda h: predictions[h]["probability"]
                ).replace("_", " ").title()
            })
        except Exception as e:
            rows.append({
                "District": d, "Heavy Rain": None, "Heatwave": None,
                "Flood": None, "Highest Risk Hazard": f"Error: {e}"
            })

    comp_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(max(8, len(selected_districts) * 1.5), 5))
    x = np.arange(len(comp_df))
    width = 0.25

    ax.bar(x - width, comp_df["Heavy Rain"], width, label="Heavy Rain", color="#3498db")
    ax.bar(x, comp_df["Heatwave"], width, label="Heatwave", color="#e67e22")
    ax.bar(x + width, comp_df["Flood"], width, label="Flood", color="#2980b9")

    ax.set_xticks(x)
    ax.set_xticklabels(comp_df["District"], rotation=20, ha="right")
    ax.set_ylabel("ML Probability")
    ax.set_ylim(0, 1)
    ax.set_title("District Comparison — Multi-Hazard Risk")
    ax.legend()
    fig.tight_layout()

    riskiest = comp_df.loc[comp_df[["Heavy Rain", "Heatwave", "Flood"]].max(axis=1).idxmax(), "District"]
    summary = f"### 📊 Comparing {len(selected_districts)} districts\n\n🔴 Highest overall risk right now: **{riskiest}**"

    return summary, comp_df, fig

In [ ]:
custom_css = """
#header-box {
    background: linear-gradient(135deg, #1e3a5f 0%, #2c5f8a 100%);
    padding: 28px 32px;
    border-radius: 12px;
    color: white !important;
    margin-bottom: 20px;
}
#header-box h1, #header-box h3, #header-box p {
    color: white !important;
    margin: 4px 0;
}
.gradio-container {
    background-color: #f7f8fa !important;
    font-family: 'Segoe UI', Roboto, Arial, sans-serif !important;
}
.gr-button-primary {
    background: #2c5f8a !important;
    border: none !important;
    font-weight: 600 !important;
}
.tabitem, .gr-box, .gr-panel {
    border-radius: 10px !important;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08) !important;
}
.dark-mode {
    background-color: #0f0f0f !important;
    color: #f0f0f0 !important;
}
.dark-mode #header-box {
    background: linear-gradient(135deg, #111 0%, #222 100%) !important;
}
.dark-mode .gr-box, .dark-mode .gr-panel, .dark-mode table {
    background-color: #1a1a1a !important;
    color: #f0f0f0 !important;
}
"""

toggle_js = """
() => {
    document.querySelector('.gradio-container').classList.toggle('dark-mode');
}
"""

theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="slate",
).set(
    body_background_fill="#f7f8fa",
    block_background_fill="white",
    block_border_width="1px",
    block_shadow="*shadow_drop_lg",
)

with gr.Blocks(title="KP Multi-Hazard AI System", theme=theme, css=custom_css) as demo:

    with gr.Row(elem_id="header-box"):
        with gr.Column(scale=8):
            gr.Markdown(
                """
# 🌍 KP Multi-Hazard AI System
### Machine Learning · Live Weather · AI Explanations
Preliminary risk assessment for Heavy Rain, Heatwave, and Flood across Khyber Pakhtunkhwa.
"""
            )
        with gr.Column(scale=2, min_width=120):
            theme_button = gr.Button("🌓 Toggle Theme", size="sm")

    gr.Markdown("⚠️ *Educational research system — not an official emergency warning service.*")

    theme_button.click(fn=None, js=toggle_js)

    with gr.Tabs():
        with gr.Tab("📍 District Assessment"):
            with gr.Row():
                with gr.Column(scale=1):
                    district = gr.Dropdown(
                        choices=list(KP_DISTRICTS.keys()),
                        value="Peshawar",
                        label="Select District"
                    )
                    run_button = gr.Button("Run Assessment", variant="primary")
                    summary = gr.Markdown()
                with gr.Column(scale=2):
                    chart = gr.Plot(label="Risk Probabilities")

            with gr.Row():
                table = gr.Dataframe(label="Model Predictions")

            with gr.Accordion("🧠 AI Explanation", open=True):
                explanation = gr.Markdown()

            run_button.click(
                fn=dashboard,
                inputs=district,
                outputs=[summary, table, chart, explanation]
            )

        with gr.Tab("📊 Compare Districts"):
            with gr.Row():
                with gr.Column(scale=1):
                    districts_multi = gr.CheckboxGroup(
                        choices=list(KP_DISTRICTS.keys()),
                        value=["Peshawar", "Nowshera", "Charsadda"],
                        label="Select Districts"
                    )
                    compare_button = gr.Button("Compare", variant="primary")
                    compare_summary = gr.Markdown()
                with gr.Column(scale=2):
                    compare_chart = gr.Plot(label="Side-by-Side Comparison")

            with gr.Row():
                compare_table = gr.Dataframe(label="Comparison Table")

            compare_button.click(
                fn=compare_districts,
                inputs=districts_multi,
                outputs=[compare_summary, compare_table, compare_chart]
            )

demo.launch(share=True, debug=True)